# CSE 4095 — Class 3: Regression and Model Error
Use AI to help write unfamiliar code, but verify and explain each modeling decision.

## 1. Load and inspect the dataset
Upload `class1_student_habits.csv`, load it as `df`, and inspect `exam_score`.

In [ ]:
# TODO: load the dataset and inspect exam_score

try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Not running in Colab. Using class1_student_habits.csv from this folder.")

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(
    "class1_student_habits.csv",
    keep_default_na=False,
    na_values=[""],
)

exam = df["exam_score"]

print("count:", exam.count())
print("mean:", round(exam.mean(), 2))
print("median:", exam.median())
print("standard deviation:", round(exam.std(), 2))
print("minimum:", exam.min())
print("maximum:", exam.max())

plt.figure(figsize=(8, 5))
plt.hist(exam, bins=range(50, 105, 5), edgecolor="black")
plt.xlabel("exam_score")
plt.ylabel("Number of students")
plt.title("Distribution of Exam Scores")
plt.show()

## Part 1 Questions

1. Why is this a regression problem? The prediction target is exam_score, and that column is a number. Each student has an actual score, such as 64.6 or 92.0, and the model is asked to estimate that number. How far the prediction is from the true score matters. A guess of 82.7 when the student scored 85 is a smaller error than a guess of 60.

2. How is this different from the Class 2 classification problem? In Class 2, exam_score was turned into a label, high_exam_score. That label was Yes when the score was 80 or higher and No when it was below 80. The classifier only had to choose one of those two categories. Here the score stays a number, and the model has to estimate the score itself.

3. What information is lost when we convert exam_score into only Yes or No? The actual score, and how far it sits from 80, both disappear. A student who scored 51.4 and a student who scored 79.9 are both No. A student who scored 80.1 and a student who scored 100 are both Yes. The Yes/No label no longer says whether someone barely missed the cutoff, landed in the middle of the class, or earned a perfect score.

4. What kind of output should a regression model produce? A number on the same scale as exam_score. For example, a regression model might predict 82.7 rather than Yes.

## Part 2 Questions

1. Describe the distribution of exam scores. There are 250 scores. The mean is 77.16, the median is 76.95, and the standard deviation is 10.33. The lowest score is 51.4 and the highest is 100. The mean and the median are almost the same, and the histogram is a single mound centered in the mid-to-high 70s. Most students score between 65 and 90. Fewer students sit in the tails: 12 scored below 60, and 27 scored 90 or above, including 3 scores of 100. The shape is roughly symmetric.

2. Are there unusually low or high values? The low end is 51.4 and the high end is 100, so the class does span a wide range, about 49 points. Those ends are still the tails of the same mound, not points sitting apart from everyone else. Using the 1.5 × IQR rule, the lower fence is about 46.3 and the upper fence is about 108.2. Every score falls inside those fences, so this dataset has no outliers by that rule. Three students scored exactly 100, which is also the highest score the scale allows.

3. Why is it useful to understand the target before trying to predict it? The summary shows the scale a prediction has to live on. A typical score is about 77, and scores usually fall within roughly 10 points of that center. Later, an error of 5 points can be compared with that spread instead of judged on its own. It also shows what a sensible prediction looks like before any model is trained: a guess near 77 is ordinary, and a guess of 40 or 110 would not match this class.

## 2. Explore one relationship
Plot `study_hours_per_week` versus `exam_score` and describe what you observe.

In [ ]:
# TODO: create the scatter plot

study_and_score = df[["study_hours_per_week", "exam_score"]].dropna()

correlation = study_and_score["study_hours_per_week"].corr(study_and_score["exam_score"])
print("Students with both values:", len(study_and_score))
print("Pearson correlation:", round(correlation, 3))

plt.figure(figsize=(8, 5))
plt.scatter(
    study_and_score["study_hours_per_week"],
    study_and_score["exam_score"],
    alpha=0.7,
)
plt.xlabel("study_hours_per_week")
plt.ylabel("exam_score")
plt.title("Study Hours per Week vs Exam Score")
plt.show()

## Part 3 Questions

1. Does there appear to be a relationship? Yes. The scatter plot trends upward, and the Pearson correlation is 0.516 for the 241 students who have both values. That is a moderate association. Students with the same study hours still have a wide range of scores, so study hours alone do not determine the exam score. Nine students are missing study_hours_per_week, so they are left out of the correlation.

2. Is the relationship positive or negative? Positive. Higher study hours are associated with higher exam scores in this dataset. This is an association in the observed rows, not evidence that studying one additional hour causes the exam score to increase by a fixed number of points. Other habits, and things this file does not record, can differ at the same time.

3. Does it appear approximately linear? Mostly. Average scores rise fairly steadily as study hours increase, from about 63 for students under 2 hours to about 85 for students between 10 and 12 hours. A straight line is a reasonable first description. The cloud is wide around that line, and the students who study the most do not keep climbing at the same rate.

4. Are there students who do not follow the general pattern? Yes. S174 studied 17.8 hours per week, the most in the class, and scored 64.8. S232 studied 5.1 hours and scored 96.5, and S085 studied 5.3 hours and scored 91.0. S073 studied 6.7 hours and scored 51.4, the lowest score in the file. Those students sit well off the upward trend.

## 3. Define X and y
Start without `quiz_average`. Explain why `student_id` is excluded.

In [ ]:
# TODO: define feature columns, X, and y

feature_columns = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "prior_programming_experience",
    "study_group",
    "commute_minutes",
]

target_column = "exam_score"

X = df[feature_columns]
y = df[target_column]

print("Predictors:")
print(list(X.columns))
print()
print("Target:", target_column)
print()
print("Left out of X:")
print(["student_id", "quiz_average"])
print()
print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

## Part 4 Explanation

student_id is excluded because it is only a label that tells the rows apart, such as S001 or S012. It is not a habit and it does not describe how the student prepared. Each id belongs to one student, so a model that used it would be memorizing that particular student instead of learning a pattern that could apply to someone new. A new student would have an id the model has never seen.

## 4. Train/test split and preprocessing
Use an 80/20 split and a pipeline that handles missing values and categorical variables.

In [ ]:
# TODO: split data and create preprocessing

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "commute_minutes",
]
categorical_features = [
    "prior_programming_experience",
    "study_group",
]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print()
print("Missing values in X:")
print(X.isna().sum())
print()
print("Prepared training shape:", X_train_prepared.shape)
print("Prepared test shape:", X_test_prepared.shape)
print()
print("Columns after preprocessing:")
print(list(preprocessor.get_feature_names_out()))

## Part 5 Questions

1. What is the training set used for? The training set is the data the model learns from. It sees each student's habits together with that student's actual exam score, and it estimates how those habits line up with the score. In this split that is 200 students, 80% of the class. The preprocessing also learns from these rows only: the median used to fill a blank, the most common category, and the mean and standard deviation used to rescale each number.

2. What is the test set used for? The test set is held back until after training. It is used to measure how far the predictions are from the real scores for students the model did not learn from. That is the check for how the model might do on a new student. Here that is 50 students, 20% of the class. random_state=42 keeps this same 200/50 division each time the notebook is run.

3. Why should the model not be trained using the test set? The test set only works as a check if the model has not already seen those students. If those 50 rows are included in training, a small error can mean the model memorized them instead of learning a pattern that works for someone new. We train on the 200 and score the model on the 50.

4. Why should different experiments use the same split when we want to compare them fairly? Each experiment should be judged on the same 50 students. If a later model, such as the one that adds quiz_average, drew a new split, a lower error might come from an easier test set rather than from the new feature. Keeping random_state=42 makes the comparison about the model, not about which students were held out.

## Part 6 Questions

1. Why do we handle numerical and categorical variables differently? The numeric columns are amounts, such as study hours, attendance, and commute minutes. A blank in one of those columns can be filled with a typical number, the training median, and the column can be rescaled. The categorical columns are words. prior_programming_experience contains None, Some, and Strong, and study_group contains Yes and No. A median does not exist for a word, and linear regression cannot multiply a word by a weight. Those columns have to become numbers first. Nine students are missing study hours, twelve are missing sleep hours, and seven are missing prior programming experience. The word None is not one of those blanks. It is a real category, meaning the student had no prior programming experience.

2. What does StandardScaler do? It rescales each numeric column using the training rows. For each value it computes (value − training mean) / training standard deviation. After that step, each numeric training column has a mean of about 0 and a standard deviation of about 1. A scaled value of 1 means one standard deviation above that column's training average. Study hours are small numbers, attendance is usually in the 70s to 100, and commute minutes can be near 0 or well over 100. Without scaling, a column with larger numbers can influence the fit more just because of its units.

3. What does one-hot encoding do? It creates a separate 0/1 column for each category. study_group becomes two columns, one for No and one for Yes. A student in a study group gets 1 in the Yes column and 0 in the No column. prior_programming_experience becomes three columns: None, Some, and Strong. The model can then learn a separate weight for each category. That is different from coding None as 0, Some as 1, and Strong as 2, which would pretend those labels are evenly spaced numbers.

4. Why should preprocessing be learned from the training data rather than the entire dataset? The median, the most common category, and the mean and standard deviation used for scaling are all learned numbers. fit records them from the 200 training students. transform applies those same numbers to the 50 test students. If those statistics were computed on all 250 rows, the test students would help decide how the training data is filled and scaled. The test set would no longer be fully unseen. A new student, at prediction time, also would not be part of that calculation. The prepared training matrix is 200 rows by 10 columns, and the prepared test matrix is 50 rows by 10 columns: the original 5 numeric columns, plus 5 columns from the categories.

## 5. Mean baseline
Predict the training-set mean for every test case and calculate baseline MAE.

In [ ]:
# TODO: calculate mean baseline and MAE

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

training_mean = y_train.mean()
baseline_predictions = [training_mean] * len(y_test)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_mse = mean_squared_error(y_test, baseline_predictions)
baseline_rmse = baseline_mse ** 0.5
baseline_r2 = r2_score(y_test, baseline_predictions)

print("Training mean exam score:", round(training_mean, 2))
print("Baseline prediction for every test student:", round(training_mean, 2))
print()
print("Baseline MAE:", round(baseline_mae, 2))
print("Baseline MSE:", round(baseline_mse, 2))
print("Baseline RMSE:", round(baseline_rmse, 2))
print("Baseline R²:", round(baseline_r2, 3))

## Part 7 Questions

Why is this useful? The baseline ignores every habit and predicts the training-set mean, 77.73, for all 50 test students. On this test set its MAE is 7.66 points, its MSE is 96.08, its RMSE is 9.80 points, and its R² is -0.091. A later regression model is only doing useful work if it beats these numbers on the same 50 students. An error around 8 points can look acceptable on its own and still be no better than telling every student to expect a score of about 78. R² is slightly below zero here because 77.73 is the training mean, while the test scores average 74.89. Predicting the training average is a little worse than predicting the test set's own average, which is the comparison R² uses.

## 6. Linear regression
Fit a preprocessing + LinearRegression pipeline using training data only.

In [ ]:
# TODO: build, fit, and predict

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

comparison = pd.DataFrame({
    "student_id": df.loc[X_test.index, "student_id"].to_numpy(),
    "actual_exam_score": y_test.to_numpy(),
    "predicted_exam_score": predictions,
})
print(comparison.head(8).round({"predicted_exam_score": 1}).to_string(index=False))

## Part 8 Questions

1. What does .fit() do? It runs the pipeline from start to finish on the 200 training students. The preprocessor records the numeric medians, the training mean and standard deviation of each numeric column, the most common category, and the category names used for the 0/1 columns. Linear regression then sees those prepared columns together with the actual exam scores and searches for an intercept and a weight for each prepared column. The fitted intercept is about 78.6. Those values are estimated from the training students only.

2. What does .predict() do? It applies the stored preprocessing to each test student, then computes intercept plus the sum of weight times prepared feature. That number is the predicted exam score. The call does not look at the test student's actual score, and it does not relearn the fills or the weights. For example, student S143 scored 72.5 and the model predicts 78.1. S007 scored 68.0 and the model predicts 68.4.

3. Why can the predicted exam score contain decimals even if the original scores are whole numbers? The prediction is a weighted sum, and that sum is not rounded to a whole point. Even if every training score were an integer, the line that best fits those points would still pass between them, so a prediction can be 82.7 rather than 82 or 83. In this file the recorded scores already include tenths, such as 72.5 and 68.0, so a decimal prediction is on the same kind of scale as the target.

## 7. Metrics
Calculate MAE, MSE, and R². Compare with the baseline.

In [ ]:
# TODO: evaluate predictions

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = mse ** 0.5
r2 = r2_score(y_test, predictions)

print("Linear regression")
print(f"MAE:  {mae:.2f}")
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.3f}")
print()
print("Mean baseline")
print(f"MAE:  {baseline_mae:.2f}")
print(f"MSE:  {baseline_mse:.2f}")
print(f"RMSE: {baseline_rmse:.2f}")
print(f"R²:   {baseline_r2:.3f}")

## Part 9 Questions

1. Which metric is easiest to explain in terms of exam-score points? MAE. The linear regression MAE is 5.77, which means the predictions differ from the actual exam scores by about 5.77 points on average. RMSE is also in points, but it is not a plain average of the misses. MSE is 69.70 squared points, which is harder to picture as a score on the exam.

2. Why does MSE penalize large errors more heavily than MAE? MSE squares each error before averaging. A miss of 2 points contributes 4, and a miss of 10 points contributes 100. In MAE those same misses contribute 2 and 10. Squaring makes the 10-point miss count much more than five times the 2-point miss, so a few large errors pull MSE up more than they pull MAE.

3. Why might RMSE be easier to interpret than MSE? RMSE is the square root of MSE, so it is back in exam-score points. An RMSE of 8.35 points can be compared with scores that run from about 51 to 100. An MSE of 69.70 cannot, because those units are squared points. RMSE still weights large errors more than MAE does, because the squaring happens before the square root.

4. Did Linear Regression outperform the mean baseline? Yes, on this same test set, on every metric. MAE falls from 7.66 to 5.77, about 1.9 points closer on average. MSE falls from 96.08 to 69.70, and RMSE falls from 9.80 to 8.35. R² rises from -0.091 to 0.209. The model is doing more than predicting 77.73 for everyone. The gain is real and still modest: a typical prediction is off by about 6 points.

5. What does the R² value tell you? An R² of 0.209 means the linear model accounts for about 21% of the variation in the test scores, compared with predicting the mean. The other 79% is still unexplained by these seven features. That matches the errors above. The model beats the baseline, whose R² was slightly below zero, and it does not account for most of the differences in exam scores.

## 8. Prediction table
Create actual, predicted, residual, and absolute-residual columns.

In [ ]:
# TODO: build prediction/error table

results = pd.DataFrame({
    "student_id": df.loc[X_test.index, "student_id"].to_numpy(),
    "actual_exam_score": y_test.to_numpy(),
    "predicted_exam_score": predictions,
})
results["residual"] = results["actual_exam_score"] - results["predicted_exam_score"]
results["absolute_residual"] = results["residual"].abs()

print(results.round(1).to_string(index=False))
print()
print(f"Mean residual: {results['residual'].mean():.2f}")
print(f"Mean absolute residual (MAE): {results['absolute_residual'].mean():.2f}")
print(f"Root mean squared residual (RMSE): {(results['residual'] ** 2).mean() ** 0.5:.2f}")

## 9. Actual vs. predicted plot
Add a perfect-prediction reference line.

In [ ]:
# TODO: actual vs predicted plot

low = min(results["actual_exam_score"].min(), results["predicted_exam_score"].min())
high = max(results["actual_exam_score"].max(), results["predicted_exam_score"].max())

plt.figure(figsize=(8, 5))
plt.scatter(results["actual_exam_score"], results["predicted_exam_score"], alpha=0.7)
plt.plot([low, high], [low, high], linestyle="--", color="black", label="actual = predicted")
plt.xlabel("Actual exam score")
plt.ylabel("Predicted exam score")
plt.title("Actual vs Predicted Exam Score")
plt.legend()
plt.show()

## Part 10 Questions

1. What would a perfect model look like on this graph? Every point would sit on the dashed line. That line is actual = predicted, so a point on it means the model named that student's score exactly. There would be no vertical distance from the line.

2. Are most points close to the diagonal? Most are in a band around it, and few sit on it. The median absolute error on these 50 students is about 4.6 points, and 42 of the 50 are within 10 points of the line. The tightest group is the middle of the plot, where actual scores are about 70 to 90. The MAE of 5.77 is the average distance from this line.

3. Where does the model appear to make larger errors? At the ends of the score range, and in one point far above the line. Students who actually scored about 50 to 70 are often predicted too high, as that group averages about 63, and the predictions average about 69. The two students who scored above 90 are predicted too low, usually in the high 70s and low 80s, so those points fall below the line. The farthest point is S174, with an actual score of 64.8 and a prediction of about 102, above both the line and the top of the exam scale. Eight of the 50 students miss by more than 10 points.

## 10. Residual plot
Plot predicted values against residuals and add a horizontal zero line.

In [ ]:
# TODO: residual plot

residuals = results["actual_exam_score"] - results["predicted_exam_score"]

plt.figure(figsize=(8, 5))
plt.scatter(results["predicted_exam_score"], residuals, alpha=0.7)
plt.axhline(0, linestyle="--", color="black", label="residual = 0")
plt.xlabel("Predicted exam score")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residual Plot")
plt.legend()
plt.show()

## Part 11 Questions

1. What does a positive residual mean? The actual score is higher than the predicted score. The model underestimated that student. On this test set, 22 of the 50 residuals are positive.

2. What does a negative residual mean? The actual score is lower than the predicted score. The model overestimated that student. The other 28 residuals are negative. S174 is the clearest case: the prediction is about 102 and the actual score is 64.8, so the residual is about -37.

3. What does a residual close to zero mean? The prediction landed close to the actual score. A residual of 0 would mean the two scores match. S007, with a residual of about -0.4, is an example of a close call.

4. Do you see any pattern in the residual plot? Through most of the plot, from predicted scores of about 61 to about 90, the points form a cloud around the zero line, with misses on both sides. That is close to what we want: no strong curve that rises or falls with the prediction. The cloud is not perfectly even. The average residual is about -0.65, so the predictions run slightly high overall. The spread is wider at the high end of the predictions because one point, a prediction near 102 with a residual near -37, sits far below the line. A few other misses of about 10 to 18 points appear on both sides of zero. That isolated point is a sign the linear model is missing something about that student. It is not a smooth bend across the rest of the test set.

## 11. Largest errors
Find and inspect the five largest absolute residuals.

In [ ]:
# TODO: inspect largest errors

error_details = results[
    ["student_id", "actual_exam_score", "predicted_exam_score", "residual"]
].copy()
error_details["absolute_error"] = (
    error_details["actual_exam_score"] - error_details["predicted_exam_score"]
).abs()

features_to_show = [
    "student_id",
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "prior_programming_experience",
    "study_group",
    "commute_minutes",
]
largest_errors = (
    error_details
    .merge(df[features_to_show], on="student_id", how="left")
    .nlargest(10, "absolute_error")
)

print(largest_errors.round(1).to_string(index=False))

## Part 12 Investigation

The table lists the 10 test students with the largest absolute errors. The two largest are discussed below. The numbers are what the row shows. The suggestions about why the model missed are hypotheses, and they do not establish a cause.

**S174:** The model predicted about 102.1, and the actual score was 64.8. The residual is about -37.3, so the model overestimated the score, and the prediction is also above the top of the exam scale. The recorded habits are 17.8 study hours per week, 98.1% attendance, Strong prior programming experience, membership in a study group, 6.9 hours of sleep, a 36-minute commute, and 0 practice problems. In the full file, 17.8 is the maximum study-hour value. The class average is about 7.5, and three-quarters of students study 9.5 hours or fewer. Zero practice problems is the bottom of that column. The class median is 16. Attendance of 98.1 is high. The class average is about 85.5. One hypothesis is that the straight line fitted on typical students keeps rising with study hours, so an extreme study-hour value, together with Strong experience and high attendance, produces a prediction above 100. Zero practice problems points the other way, and in this row it was not enough to bring the prediction down to 64.8. That is a hypothesis about an unusual combination and about extrapolation. It does not show that the long study hours caused the low score, or that skipping practice problems caused it.

**S109:** The model predicted about 82.2, and the actual score was 100. The residual is about 17.8, so the model underestimated a perfect score. The recorded habits are 10.1 study hours, 7.3 hours of sleep, 91.6% attendance, 25 practice problems, prior experience of None, membership in a study group, and a commute of 0 minutes. Study hours are above the class average of about 7.5 and just above the 75th percentile of 9.5. Practice problems are above the 75th percentile of 23. Attendance is high. Sleep is near the class average of about 7. The prediction of 82.2 is a few points above the training mean of 77.73, so the model did treat this row as somewhat above average. One hypothesis is that prior experience of None pulled the prediction down relative to a student with the same hours, attendance, and practice problems and a Strong label. Another is that something this model never sees accounts for the rest of the gap from 82.2 to 100. This student's quiz average is 94.1, and quiz average was left out of the model. Either idea is a hypothesis. The row shows a perfect exam score next to above-average study habits and no recorded prior experience. It does not show that the None label caused the miss.

In [ ]:
# Coefficients from the fitted linear regression, after preprocessing.

regressor = model.named_steps["regressor"]
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

coefficients = pd.DataFrame({
    "transformed_feature": feature_names,
    "coefficient": regressor.coef_,
})
coefficients = coefficients.sort_values("coefficient")

print(f"Intercept: {regressor.intercept_:.2f}")
print(coefficients.round(2).to_string(index=False))

plt.figure(figsize=(8, 5))
plt.barh(coefficients["transformed_feature"], coefficients["coefficient"])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coefficient")
plt.title("Linear Regression Coefficients after Preprocessing")
plt.tight_layout()
plt.show()

## Part 13 Questions

Each numeric coefficient is the change in the predicted score associated with a one-standard-deviation increase in that column on the training data. For study hours, one training standard deviation is about 2.9 hours. The category columns are 0/1 indicators. The intercept is about 78.56, close to the training-set mean, which fits a model whose numeric inputs are centered at zero.

1. Which features have positive coefficients? Study hours (5.93), Strong prior programming experience (5.06), practice problems (3.17), attendance (1.98), sleep (1.33), commute minutes (0.53), and study group = No (0.12).

2. Which features have negative coefficients? Prior experience = None (-4.87), prior experience = Some (-0.19), and study group = Yes (-0.12).

3. Which coefficients have relatively large magnitudes? Study hours, Strong, and None are the large ones, around 5 points in absolute value. Practice problems is next, at 3.17. Attendance and sleep are smaller, at 1.98 and 1.33. Commute, Some, and both study-group columns are under 1 in absolute value. In this fit, study group barely moves the prediction.

4. What does a positive coefficient mean in this fitted model? It means the fitted equation associates a higher value of that prepared column with a higher predicted exam score, holding the other prepared columns fixed. A positive attendance coefficient means the model associates higher attendance with a higher predicted score. It does not prove that raising attendance by itself causes that change in the exam score. The same reading applies to study hours: the 5.93 is the associated change for a one-standard-deviation increase in the fitted equation, not a measured effect of assigning someone extra study time. For the category columns, Strong's positive weight and None's negative weight mean the equation assigns a higher prediction to Strong than to None when the other modeled features are held fixed. That is still an association in this sample, not evidence that prior experience causes the score difference.

## 12. Add quiz_average
Repeat using the same split. Compare metrics and discuss whether the feature is available at prediction time.

My Hypothesis: I expect adding quiz_average to reduce prediction error because a quiz average is a record of how the student has already performed in this course. In this file, higher quiz averages tend to come with higher exam scores. If that association holds on the same 50 test students, MAE and RMSE should fall, and R² should rise, compared with the original linear regression.

In [ ]:
# TODO: quiz_average experiment

quiz_features = feature_columns + ["quiz_average"]
X_quiz = df[quiz_features]
X_quiz_train = X_quiz.loc[X_train.index]
X_quiz_test = X_quiz.loc[X_test.index]

quiz_numeric = numeric_features + ["quiz_average"]
quiz_preprocessor = ColumnTransformer(transformers=[
    ("numeric", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), quiz_numeric),
    ("categorical", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

quiz_model = Pipeline(steps=[
    ("preprocessor", quiz_preprocessor),
    ("regressor", LinearRegression()),
])
quiz_model.fit(X_quiz_train, y_train)
quiz_predictions = quiz_model.predict(X_quiz_test)

quiz_mae = mean_absolute_error(y_test, quiz_predictions)
quiz_rmse = mean_squared_error(y_test, quiz_predictions) ** 0.5
quiz_r2 = r2_score(y_test, quiz_predictions)

comparison = pd.DataFrame({
    "model": [
        "Mean baseline",
        "Linear regression",
        "Linear regression + quiz_average",
    ],
    "MAE": [baseline_mae, mae, quiz_mae],
    "RMSE": [baseline_rmse, rmse, quiz_rmse],
    "R2": [baseline_r2, r2, quiz_r2],
})
print("Same 200 training students and 50 test students")
print(f"Correlation of quiz_average and exam_score: {df['quiz_average'].corr(df['exam_score']):.3f}")
print()
print(comparison.round(3).to_string(index=False))

## Part 14 Questions

1. Did MAE improve? Yes. On the same 50 test students, MAE falls from 5.77 for the original linear regression to 4.75 with quiz_average. Both are below the mean baseline MAE of 7.66. The typical absolute miss shrinks by about one point.

2. Did RMSE improve? Yes. RMSE falls from 8.35 to 6.23. The baseline RMSE is 9.80. The larger drop in RMSE than in MAE means some of the big misses got smaller, because RMSE gives those misses extra weight.

3. Did R² improve? Yes. R² rises from 0.209 to 0.559. The baseline R² is -0.091. With quiz_average, the model accounts for about 56% of the variation in the test scores, compared with about 21% from the habit features alone.

4. Was the hypothesis supported? Yes. The expectation was that MAE and RMSE would fall and R² would rise, because higher quiz averages come with higher exam scores in this file. The correlation between those two columns is 0.769. The comparison on the original 200 training students and 50 test students moved in that direction. The predictions are still off by about 4.75 points on average, so quiz average reduces the error and does not remove it.

5. Would quiz_average be available when the prediction is supposed to be made? It depends on the decision. Every student in this file has a quiz average, and there are no blanks in that column. If the prediction is made after the quizzes are graded and before the final exam, quiz_average is a legitimate feature. It is known at that time, and it is not the exam score itself. If the prediction has to be made at the start of the term, before any quiz has happened, the average would not exist yet. A model that needs it cannot be used for that earlier decision, even though the column is strongly associated with the exam. A feature can be highly predictive and still be the wrong input for a particular moment.

## 13. Nonlinear model
Try one nonlinear regressor, such as DecisionTreeRegressor, using the same split.

In [ ]:
# TODO: nonlinear model experiment

from sklearn.ensemble import RandomForestRegressor

# Trees split on thresholds, so StandardScaler is not needed here.
# Missing values and category labels still have to be handled.
forest_preprocessor = ColumnTransformer(transformers=[
    ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ("categorical", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

forest_model = Pipeline(steps=[
    ("preprocessor", forest_preprocessor),
    ("regressor", RandomForestRegressor(random_state=42)),
])
forest_model.fit(X_train, y_train)
forest_predictions = forest_model.predict(X_test)

forest_mae = mean_absolute_error(y_test, forest_predictions)
forest_rmse = mean_squared_error(y_test, forest_predictions) ** 0.5
forest_r2 = r2_score(y_test, forest_predictions)

model_comparison = pd.DataFrame({
    "model": ["Mean baseline", "Linear regression", "Random forest"],
    "MAE": [baseline_mae, mae, forest_mae],
    "RMSE": [baseline_rmse, rmse, forest_rmse],
    "R2": [baseline_r2, r2, forest_r2],
})
print("Same 200 training students, 50 test students, and seven features")
print(model_comparison.round(3).to_string(index=False))

## Part 15 Questions

1. Which model had the lowest MAE? Linear regression, at 5.77. The random forest MAE is 6.25, and the mean baseline is 7.66.

2. Which had the lowest RMSE? The random forest, at 7.94. Linear regression is 8.35, and the baseline is 9.80.

3. Which had the highest R²? The random forest, at 0.284. Linear regression is 0.209, and the baseline is -0.091.

4. Did the more complicated model necessarily perform better? No. On this dataset and this particular train/test split, the random forest has the lower RMSE and the higher R², and linear regression has the lower MAE. The forest is closer on the squared-error summaries and farther off on the average absolute miss. Both beat the baseline of predicting 77.73 for everyone. That is a result for these 50 test students, not a general ranking of the two algorithms. A later comparison with cross-validation would check more than one split.

## Part 16 — Does RandomForestRegressor require StandardScaler?

No, a random forest is a set of decision trees, and a tree split asks whether a feature is on one side of a cutoff. One example is study_hours_per_week < 8.5. StandardScaler replaces every study-hour value with (value − training mean) / training standard deviation. The same shift and the same stretch are applied to every student, so the student who studied the least is still the least, and the student who studied the most is still the most. The cutoff number changes, and the same students still fall on each side of the split. That is why the forest above fills missing values and encodes the categories, and leaves the numeric columns unscaled. Linear regression is different, since each coefficient multiplies the size of the number, so columns on larger scales can dominate the sum unless they are rescaled.

## 14. Your experiment and AI Reflection
State your expectation before running the experiment. Then summarize the evidence and reflect on one AI suggestion you evaluated.

## My Hypothesis

My Hypothesis: I expect filling the missing numeric values with the training-set mean, instead of the training-set median, to leave the test error almost unchanged, because the only numeric columns with blanks are study hours and sleep hours, and those two columns are not strongly skewed. Their means and medians should be close, and only a few test students are missing either value. A small change in the fill, applied to a few rows, should not move MAE, RMSE, or R² in a meaningful way. The category fill stays the most common category, so this comparison is only about the numeric strategy.

In [ ]:
# TODO: your experiment

# on this same split, does a training-mean fill change the original linear model's test error 
# compared with a training-median fill?
mean_preprocessor = ColumnTransformer(transformers=[
    ("numeric", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("categorical", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

mean_model = Pipeline(steps=[
    ("preprocessor", mean_preprocessor),
    ("regressor", LinearRegression()),
])
mean_model.fit(X_train, y_train)
mean_predictions = mean_model.predict(X_test)

mean_mae = mean_absolute_error(y_test, mean_predictions)
mean_rmse = mean_squared_error(y_test, mean_predictions) ** 0.5
mean_r2 = r2_score(y_test, mean_predictions)

for column in ["study_hours_per_week", "sleep_hours_per_night"]:
    print(
        column,
        "train mean", round(X_train[column].mean(), 3),
        "train median", round(X_train[column].median(), 3),
        "missing in test", int(X_test[column].isna().sum()),
    )

fill_comparison = pd.DataFrame({
    "model": ["Linear regression, median fill", "Linear regression, mean fill"],
    "MAE": [mae, mean_mae],
    "RMSE": [rmse, mean_rmse],
    "R2": [r2, mean_r2],
})
print()
print(fill_comparison.round(4).to_string(index=False))

## Part 17: My Results

On the training rows, the study-hour mean is 7.685 and the median is 7.6. The sleep-hour mean is 6.967 and the median is 7.0. Three test students are missing study hours, and three are missing sleep hours.

| Model | MAE | RMSE | R² |
|---|---|---|---|
| Linear regression, median fill | 5.7740 | 8.3485 | 0.2085 |
| Linear regression, mean fill | 5.7749 | 8.3498 | 0.2083 |

The evidence supports my hypothesis, as MAE rises by less than 0.001, RMSE rises by about 0.001, and R² falls by about 0.0002. At the two-decimal reporting used earlier, both models are MAE 5.77 and RMSE 8.35. Mean imputation is not an improvement on this split. It is the same prediction for practical purposes. I did not try another fill after seeing this table. 

One AI suggestion I evaluated was to try several imputation strategies and keep whichever one scored best on the test set. I did not do that. Using the 50 test students to shop for a better score would turn the test set into a tuning set, and a small gain could be luck on these particular students. The comparison above was specified before it was run: same model, same split, median fill versus mean fill.

## Part 18 — AI Reflection

AI helped me interpret an unexpected result from the mean baseline. I asked how to score a model that predicts the training-set mean for every test student, and what MAE, MSE, RMSE, and R² say about that score. The suggestion was to calculate all four on the same 50 test students, and to treat a negative R² as a real outcome rather than a bug. I accepted that. The notebook reports a baseline MAE of 7.66, an RMSE of 9.80, and an R² of -0.091. I checked those prints against the training mean of 77.73 and the test-set average of 74.89. I did not change the formulas after seeing the negative sign. What I learned is that MAE is the average miss in exam points, MSE squares the misses before averaging, and RMSE takes the square root so the result is in points again. R² compares the predictions with predicting the test set's own mean. Predicting the training mean can score slightly below zero when that mean is not the test mean. The linear model later reached an R² of 0.209, which is the same comparison showing a real improvement over the baseline.

One suggestion I questioned was that the random forest could skip StandardScaler. I questioned it because the linear pipeline uses scaling, and the comparison was supposed to keep the preprocessing where appropriate. The explanation was a split such as study_hours_per_week < 8.5. After standardization the cutoff number changes, and the same students still fall on each side, because scaling does not change their order. That suggestion turned out to be right. I accepted it for the forest and kept the median fill and the 0/1 category columns. I verified it by fitting that pipeline on the same 200 training students and scoring the same 50 test students. The forest ran and produced MAE 6.25, RMSE 7.94, and R² 0.284. I also did not accept a broader claim that the forest is the better algorithm. On this split it beat linear regression on RMSE and R², and linear regression still had the lower MAE, 5.77 against 6.25.